In [1]:
import pandas as pd
import world_bank_data as wb

print("OK")

OK


In [2]:
COUNTRIES = ["AR", "DE", "BR", "MX"]

INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "FP.CPI.TOTL.ZG":    "inflation",
    "SL.UEM.TOTL.ZS":    "unemployment",
    "GC.DOD.TOTL.GD.ZS": "debt_to_gdp",
    "BX.KLT.DINV.WD.GD.ZS": "fdi_inflow",
}

YEAR_START = 2004
YEAR_END   = 2023

print("OK")

OK


In [3]:
dataframes = []

for code, nombre in INDICATORS.items():
    print(f"Descargando: {nombre}...")
    serie = wb.get_series(
        code,
        country=COUNTRIES,
        mrv=YEAR_END - YEAR_START + 1,
        simplify_index=True
    )
    df = serie.reset_index()
    df.columns = ["country", "year", "value"]
    df["indicator"] = nombre
    dataframes.append(df)

raw = pd.concat(dataframes, ignore_index=True)
print(f"\nOK — {len(raw)} filas descargadas")

Descargando: gdp_growth...
Descargando: inflation...
Descargando: unemployment...
Descargando: debt_to_gdp...
Descargando: fdi_inflow...

OK — 400 filas descargadas


In [4]:
pivot = raw.pivot_table(
    index=["country", "year"],
    columns="indicator",
    values="value"
).reset_index()

pivot.columns.name = None
print("OK")
print(pivot.columns.tolist())

OK
['country', 'year', 'debt_to_gdp', 'fdi_inflow', 'gdp_growth', 'inflation', 'unemployment']


In [5]:
COUNTRY_NAMES = {
    "AR": "Argentina",
    "DE": "Germany",
    "BR": "Brazil",
    "MX": "Mexico"
}

arg = pivot[pivot["country"] == "Argentina"][["year", "inflation"]].sort_values("year")
print("Inflación Argentina por año:")
print(arg.to_string(index=False))

Inflación Argentina por año:
year  inflation
2005        NaN
2006        NaN
2007        NaN
2008        NaN
2009        NaN
2010        NaN
2011        NaN
2012        NaN
2013        NaN
2014        NaN
2015        NaN
2016        NaN
2017        NaN
2018  34.277224
2019  53.548304
2020  42.015095
2021  48.409379
2022  72.430758
2023 133.488936
2024 219.883929
2025        NaN


In [6]:
output_path = "../data/world_bank_raw.csv"
pivot.to_csv(output_path, index=False)
print(f"Guardado OK")

Guardado OK


In [7]:
import requests

# Dataset del FMI - World Economic Outlook
# Inflación argentina histórica (fuente citable en el README)
url = "https://www.imf.org/external/datamapper/api/v1/PCPIPCH/ARG?periods=2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017"

response = requests.get(url)
data = response.json()

imf_inflation = data["values"]["PCPIPCH"]["ARG"]

print("Inflación Argentina según FMI:")
for year, value in sorted(imf_inflation.items()):
    print(f"  {year}: {value:.2f}%")

Inflación Argentina según FMI:
  1998: 0.90%
  1999: -1.20%
  2000: -0.90%
  2001: -1.10%
  2002: 25.90%
  2003: 13.40%
  2004: 4.40%
  2005: 9.60%
  2006: 10.90%
  2007: 8.80%
  2008: 8.60%
  2009: 6.30%
  2010: 10.50%
  2011: 9.80%
  2012: 10.00%
  2013: 10.60%
  2017: 25.70%
  2018: 34.30%
  2019: 53.50%
  2020: 42.00%
  2021: 48.40%
  2022: 72.40%
  2023: 133.50%
  2024: 219.90%
  2025: 41.90%
  2026: 30.40%
  2027: 15.70%
  2028: 9.60%
  2029: 7.50%
  2030: 7.50%
  2031: 7.50%


In [8]:
# Convertir datos FMI a DataFrame
imf_df = pd.DataFrame([
    {"year": int(y), "inflation_imf": v}
    for y, v in imf_inflation.items()
])

# Filtrar solo 2004-2023
imf_df = imf_df[(imf_df["year"] >= 2004) & (imf_df["year"] <= 2023)]

print("Datos FMI filtrados (2004-2023):")
print(imf_df.to_string(index=False))

Datos FMI filtrados (2004-2023):
 year  inflation_imf
 2004            4.4
 2005            9.6
 2006           10.9
 2007            8.8
 2008            8.6
 2009            6.3
 2010           10.5
 2011            9.8
 2012           10.0
 2013           10.6
 2017           25.7
 2018           34.3
 2019           53.5
 2020           42.0
 2021           48.4
 2022           72.4
 2023          133.5


In [11]:
# Convertir year a int en ambos dataframes antes del merge
arg_df["year"] = arg_df["year"].astype(int)
imf_df["year"] = imf_df["year"].astype(int)

# Separar Argentina del resto
arg_mask = pivot["country"] == "Argentina"
arg_df = pivot[arg_mask].copy()
rest_df = pivot[~arg_mask].copy()

# Convertir year a int
arg_df["year"] = arg_df["year"].astype(int)

# Mergear con datos del FMI
arg_df = arg_df.merge(imf_df, on="year", how="left")

# Completar inflación: usar World Bank donde hay, FMI donde no hay
arg_df["inflation"] = arg_df["inflation"].fillna(arg_df["inflation_imf"])

# Eliminar columna auxiliar
arg_df = arg_df.drop(columns=["inflation_imf"])

print("Inflación Argentina después del merge:")
print(arg_df[["year", "inflation"]].to_string(index=False))

Inflación Argentina después del merge:
 year  inflation
 2005   9.600000
 2006  10.900000
 2007   8.800000
 2008   8.600000
 2009   6.300000
 2010  10.500000
 2011   9.800000
 2012  10.000000
 2013  10.600000
 2014        NaN
 2015        NaN
 2016        NaN
 2017  25.700000
 2018  34.277224
 2019  53.548304
 2020  42.015095
 2021  48.409379
 2022  72.430758
 2023 133.488936
 2024 219.883929
 2025        NaN


In [12]:
# Ordenar por año e interpolar linealmente los NaN restantes
arg_df = arg_df.sort_values("year")
arg_df["inflation"] = arg_df["inflation"].interpolate(method="linear")

# Verificar que no quede ningún NaN en inflación
nulos = arg_df["inflation"].isna().sum()
print(f"NaN restantes en inflación AR: {nulos}")
print("\nInflación Argentina final:")
print(arg_df[["year", "inflation"]].to_string(index=False))

NaN restantes en inflación AR: 0

Inflación Argentina final:
 year  inflation
 2005   9.600000
 2006  10.900000
 2007   8.800000
 2008   8.600000
 2009   6.300000
 2010  10.500000
 2011   9.800000
 2012  10.000000
 2013  10.600000
 2014  14.375000
 2015  18.150000
 2016  21.925000
 2017  25.700000
 2018  34.277224
 2019  53.548304
 2020  42.015095
 2021  48.409379
 2022  72.430758
 2023 133.488936
 2024 219.883929
 2025 219.883929


In [14]:
# Convertir year a int en ambos antes de unir
arg_df["year"] = arg_df["year"].astype(int)
rest_df["year"] = rest_df["year"].astype(int)

# Reunir Argentina corregida con el resto
pivot_final = pd.concat([arg_df, rest_df], ignore_index=True)
pivot_final = pivot_final.sort_values(["country", "year"]).reset_index(drop=True)

# Filtrar solo 2004-2023
pivot_final = pivot_final[(pivot_final["year"] >= 2004) & (pivot_final["year"] <= 2023)]

# Guardar
output_path = "../data/world_bank_raw.csv"
pivot_final.to_csv(output_path, index=False)

print("Dataset final guardado.")
print(f"Shape: {pivot_final.shape}")
print(f"\nNulos por columna:")
print(pivot_final.isnull().sum())

Dataset final guardado.
Shape: (76, 7)

Nulos por columna:
country          0
year             0
debt_to_gdp     57
fdi_inflow       0
gdp_growth       4
inflation        3
unemployment     4
dtype: int64


In [15]:
print("Nulos de debt_to_gdp por país:")
print(pivot_final.groupby("country")["debt_to_gdp"].apply(lambda x: x.isna().sum()))

print("\nEjemplo Argentina debt_to_gdp:")
print(pivot_final[pivot_final["country"] == "Argentina"][["year", "debt_to_gdp"]].to_string(index=False))

Nulos de debt_to_gdp por país:
country
Argentina    19
Brazil        5
Germany      19
Mexico       14
Name: debt_to_gdp, dtype: int64

Ejemplo Argentina debt_to_gdp:
 year  debt_to_gdp
 2005          NaN
 2006          NaN
 2007          NaN
 2008          NaN
 2009          NaN
 2010          NaN
 2011          NaN
 2012          NaN
 2013          NaN
 2014          NaN
 2015          NaN
 2016          NaN
 2017          NaN
 2018          NaN
 2019          NaN
 2020          NaN
 2021          NaN
 2022          NaN
 2023          NaN


In [16]:
# Bajar deuda pública del FMI para los 4 países
paises_fmi = {"AR": "ARG", "DE": "DEU", "BR": "BRA", "MX": "MEX"}
nombres_fmi = {"ARG": "Argentina", "DEU": "Germany", "BRA": "Brazil", "MEX": "Mexico"}

debt_frames = []

for code_wb, code_fmi in paises_fmi.items():
    url = f"https://www.imf.org/external/datamapper/api/v1/GGXWDG_NGDP/{code_fmi}"
    response = requests.get(url)
    data = response.json()
    
    valores = data["values"]["GGXWDG_NGDP"][code_fmi]
    
    df_temp = pd.DataFrame([
        {"country": nombres_fmi[code_fmi], "year": int(y), "debt_to_gdp_fmi": float(v)}
        for y, v in valores.items()
    ])
    debt_frames.append(df_temp)

debt_df = pd.concat(debt_frames, ignore_index=True)
debt_df = debt_df[(debt_df["year"] >= 2004) & (debt_df["year"] <= 2023)]

print("Deuda/BIP por país (primeras filas):")
print(debt_df.groupby("country").head(3).to_string(index=False))
print(f"\nNulos: {debt_df['debt_to_gdp_fmi'].isna().sum()}")

Deuda/BIP por país (primeras filas):
  country  year  debt_to_gdp_fmi
Argentina  2004            117.9
Argentina  2005             80.3
Argentina  2006             70.8
  Germany  2004             65.0
  Germany  2005             67.1
  Germany  2006             66.4
   Brazil  2004             68.0
   Brazil  2005             67.0
   Brazil  2006             64.6
   Mexico  2004             38.9
   Mexico  2005             36.8
   Mexico  2006             35.8

Nulos: 0


In [17]:
# Mergear deuda del FMI al dataset principal
pivot_final = pivot_final.merge(debt_df, on=["country", "year"], how="left")

# Reemplazar columna vieja con la del FMI
pivot_final["debt_to_gdp"] = pivot_final["debt_to_gdp_fmi"]
pivot_final = pivot_final.drop(columns=["debt_to_gdp_fmi"])

# Guardar
output_path = "../data/world_bank_raw.csv"
pivot_final.to_csv(output_path, index=False)

print("Guardado OK")
print(f"Shape: {pivot_final.shape}")
print(f"\nNulos por columna:")
print(pivot_final.isnull().sum())

Guardado OK
Shape: (76, 7)

Nulos por columna:
country         0
year            0
debt_to_gdp     0
fdi_inflow      0
gdp_growth      4
inflation       3
unemployment    4
dtype: int64


In [18]:
# Interpolar por país para no mezclar valores entre países
cols_a_interpolar = ["gdp_growth", "inflation", "unemployment"]

grupos = []
for country in pivot_final["country"].unique():
    subset = pivot_final[pivot_final["country"] == country].copy()
    subset = subset.sort_values("year")
    subset[cols_a_interpolar] = subset[cols_a_interpolar].interpolate(method="linear")
    grupos.append(subset)

pivot_final = pd.concat(grupos, ignore_index=True)
pivot_final = pivot_final.sort_values(["country", "year"]).reset_index(drop=True)

# Guardar
output_path = "../data/world_bank_raw.csv"
pivot_final.to_csv(output_path, index=False)

print("Guardado OK")
print(f"\nNulos restantes:")
print(pivot_final.isnull().sum())

Guardado OK

Nulos restantes:
country         0
year            0
debt_to_gdp     0
fdi_inflow      0
gdp_growth      4
inflation       3
unemployment    4
dtype: int64


In [19]:
for col in ["gdp_growth", "inflation", "unemployment"]:
    print(f"\n--- {col} ---")
    nulos = pivot_final[pivot_final[col].isna()][["country", "year", col]]
    print(nulos.to_string(index=False))


--- gdp_growth ---
  country  year  gdp_growth
Argentina  2005         NaN
   Brazil  2005         NaN
  Germany  2005         NaN
   Mexico  2005         NaN

--- inflation ---
country  year  inflation
 Brazil  2005        NaN
Germany  2005        NaN
 Mexico  2005        NaN

--- unemployment ---
  country  year  unemployment
Argentina  2005           NaN
   Brazil  2005           NaN
  Germany  2005           NaN
   Mexico  2005           NaN


In [20]:
cols_a_interpolar = ["gdp_growth", "inflation", "unemployment"]

grupos = []
for country in pivot_final["country"].unique():
    subset = pivot_final[pivot_final["country"] == country].copy()
    subset = subset.sort_values("year")
    subset[cols_a_interpolar] = subset[cols_a_interpolar].interpolate(method="linear")
    # Para extremos sin vecino: usar el valor más cercano disponible
    subset[cols_a_interpolar] = subset[cols_a_interpolar].ffill().bfill()
    grupos.append(subset)

pivot_final = pd.concat(grupos, ignore_index=True)
pivot_final = pivot_final.sort_values(["country", "year"]).reset_index(drop=True)

pivot_final.to_csv("../data/world_bank_raw.csv", index=False)

print("Guardado OK")
print(f"\nNulos restantes:")
print(pivot_final.isnull().sum())

Guardado OK

Nulos restantes:
country         0
year            0
debt_to_gdp     0
fdi_inflow      0
gdp_growth      0
inflation       0
unemployment    0
dtype: int64
